In [ ]:
from pyexpat import features
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

In [ ]:
df = pd.read_csv("/Users/methmi/Desktop/Titanic-Dataset.csv")

print(df.head())
print(df.shape)

In [ ]:
print("Data Info:")
print(df.info())

print("Statistical Summary:")
print(df.describe())

In [ ]:
print("Missing Values:")
print(df.isnull().sum())

print("Duplicate Rows:")
print(df.duplicated().sum())

In [ ]:
df = df.drop_duplicates()

df['Age'].fillna(df['Age'].median(), inplace=True)
df['Embarked'].fillna(df['Embarked'].mode()[0], inplace=True)

df.drop('Cabin', axis=1, inplace=True)

print("Missing Values After Handling:")
print(df.isnull().sum())

In [ ]:
sns.countplot(x='Survived', data=df)
plt.title("Survival Count")
plt.show()

In [ ]:
sns.countplot(x='Sex', hue='Survived', data=df)
plt.title("Survival by Gender")
plt.show()

In [ ]:
sns.countplot(x='Pclass', hue='Survived', data=df)
plt.title("Passenger Class vs Survival")
plt.show()

In [ ]:
sns.histplot(df['Age'], bins=20, kde=True)
plt.title("Age Distribution")
plt.show()

In [ ]:
sns.boxplot(x=df['Fare'])
plt.title("Fare Outliers")
plt.show()

In [ ]:
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap="coolwarm")
plt.title("Correlation Heatmap")
plt.show()

In [ ]:
le = LabelEncoder()

df['Sex'] = le.fit_transform(df['Sex'])
df['Embarked'] = le.fit_transform(df['Embarked'])

df_features = df.drop(columns=["PassengerId", "Name", "Ticket"], errors="ignore")

In [ ]:
scaler = MinMaxScaler()

df_normalized = pd.DataFrame(
    scaler.fit_transform(df.select_dtypes(include=np.number)),
    columns=df.select_dtypes(include=np.number).columns
)

print("Normalized Data:")
print(df_normalized.head())

In [ ]:
df_features_numeric = df_features.select_dtypes(include=np.number)

standard = StandardScaler()

df_standardized = pd.DataFrame(
    standard.fit_transform(df_features_numeric),
    columns=df_features_numeric.columns
)

print("Standardized Data:")
print(df_standardized.head())

In [ ]:
print("Before Outlier Handling:")
print(df_features.describe())

In [ ]:
def cap_outliers_iqr(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    df[column] = np.where(df[column] < lower, lower, df[column])
    df[column] = np.where(df[column] > upper, upper, df[column])

    return df

In [ ]:
numeric_cols = df_features.select_dtypes(include=np.number).columns

for col in numeric_cols:
    df_features = cap_outliers_iqr(df_features, col)

print("After Outlier Handling:")
print(df_features.describe())

In [ ]:
X = df_features.drop("Survived", axis=1)
y = df_features["Survived"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

In [ ]:
rf = RandomForestClassifier(random_state=42)

rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)

print(y_pred)

In [ ]:
print("Accuracy Score:")
print(accuracy_score(y_test, y_pred))

print("F1 Score:")
print(f1_score(y_test, y_pred))

print("Classification Report:")
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred)

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

In [ ]:
print("Classification Completed Successfully!")